# CSRNet Fine-Tuning - Quick Test & Debug Notebook

This notebook is for quick testing and debugging the CSRNet fine-tuning pipeline.

## Contents:
1. **Environment Setup** - Verify imports and paths
2. **Dataset Verification** - Check dataset loading
3. **Density Map Inspection** - Visualize density maps
4. **Model Testing** - Test model forward pass
5. **Training Test** - Quick training test (1-2 epochs)
6. **Evaluation Test** - Test evaluation script

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import h5py

# Add project root to path
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

print(f"🐍 Python: {sys.version}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🖥️  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Dataset Verification

In [ ]:
# Check if ShanghaiTech dataset exists
dataset_root = project_root / "ml" / "datasets" / "raw" / "ShanghaiTech" / "ShanghaiTech"
print(f"📁 Dataset root: {dataset_root}")
print(f"✅ Exists: {dataset_root.exists()}")

# Check Part A structure
part_a = dataset_root / "part_A"
print(f"\n📁 Part A:")
print(f"   Train images: {(part_a / 'train_data' / 'images').exists()}")
print(f"   Train GT: {(part_a / 'train_data' / 'ground-truth').exists()}")
print(f"   Test images: {(part_a / 'test_data' / 'images').exists()}")
print(f"   Test GT: {(part_a / 'test_data' / 'ground-truth').exists()}")

# Count images
train_images = list((part_a / 'train_data' / 'images').glob('*.jpg'))
test_images = list((part_a / 'test_data' / 'images').glob('*.jpg'))
print(f"\n📊 Image counts:")
print(f"   Train: {len(train_images)}")
print(f"   Test: {len(test_images)}")

## 3. Test Density Map Generation

In [ ]:
# Test density map generation on ONE image
from ml.src.csrnet.training.generate_density_maps import generate_density_map_from_mat

# Pick first training image
test_img_path = train_images[0]
test_img_name = test_img_path.stem

print(f"🖼️  Test image: {test_img_name}")

# Load image
img = Image.open(test_img_path)
print(f"   Image size: {img.size}")

# Find annotation file
gt_path = part_a / 'train_data' / 'ground-truth' / f'GT_{test_img_name}.mat'
print(f"   GT file: {gt_path.exists()}")

# Generate density map
if gt_path.exists():
    density = generate_density_map_from_mat(str(gt_path), (img.size[1], img.size[0]))
    print(f"   Density shape: {density.shape}")
    print(f"   Count: {density.sum():.2f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img)
    axes[0].set_title(f'Image: {test_img_name}')
    axes[0].axis('off')
    
    im = axes[1].imshow(density, cmap='jet')
    axes[1].set_title(f'Density Map (Count: {density.sum():.0f})')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()

## 4. Test Dataset Loading

In [ ]:
# Import dataset class
from ml.src.csrnet.training.dataset import ShanghaiTechPartA

# Check if density maps exist
density_root = project_root / "ml" / "datasets" / "processed"
train_density_dir = density_root / "part_A" / "train_data" / "density_maps"

if not train_density_dir.exists():
    print("⚠️  Density maps not found! Run generate_density_maps.py first:")
    print("   python ml/src/csrnet/training/generate_density_maps.py")
else:
    density_files = list(train_density_dir.glob('*.h5'))
    print(f"✅ Found {len(density_files)} density maps")
    
    # Test dataset loading
    try:
        data = ShanghaiTechPartA(
            dataset_root=str(dataset_root),
            density_root=str(density_root),
            batch_size=2,
            num_workers=0
        )
        
        train_loader = data.get_train_loader()
        print(f"✅ Train loader created: {len(train_loader)} batches")
        
        # Load one batch
        img, density, count = next(iter(train_loader))
        print(f"\n📦 Batch shapes:")
        print(f"   Images: {img.shape}")
        print(f"   Density: {density.shape}")
        print(f"   Counts: {count.tolist()}")
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")

## 5. Test Model Loading

In [ ]:
from ml.src.models.csrnet.csrnet import load_csrnet

# Load model
checkpoint_path = project_root / "ml" / "checkpoints" / "csrnet.pth"
print(f"📥 Loading checkpoint: {checkpoint_path}")

if checkpoint_path.exists():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = load_csrnet(str(checkpoint_path), device=device)
    
    print(f"✅ Model loaded on {device}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Parameters: {total_params:,}")
    
    # Test forward pass
    if 'img' in locals():
        with torch.no_grad():
            test_input = img[:1].to(device)  # Take first image
            output = model(test_input)
            print(f"\n🔍 Forward pass test:")
            print(f"   Input shape: {test_input.shape}")
            print(f"   Output shape: {output.shape}")
            print(f"   Predicted count: {output.sum().item():.2f}")
else:
    print(f"❌ Checkpoint not found: {checkpoint_path}")

## 6. Quick Training Test (Optional - Run if you want to test training loop)

In [ ]:
# Quick 2-epoch training test
# WARNING: This will start actual training! Comment out if not needed.

# Uncomment to test:
# !python ml/src/csrnet/training/train.py --config ml/csrnet_config.yaml

print("⚠️  Quick training test disabled by default")
print("   Uncomment the line above to test training loop")

## 7. Configuration Check

In [ ]:
import yaml

# Load training config
config_path = project_root / "ml" / "csrnet_config.yaml"
print(f"📄 Config: {config_path}")

if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("\n⚙️  Training Configuration:")
    print(f"   Epochs: {config['training']['hyperparameters']['epochs']}")
    print(f"   Batch size: {config['training']['hyperparameters']['batch_size']}")
    print(f"   Learning rate: {config['training']['hyperparameters']['learning_rate']}")
    print(f"   Checkpoint dir: {config['checkpointing']['save_dir']}")
    print(f"   Log dir: {config['logging']['log_dir']}")
else:
    print("❌ Config file not found!")

## 8. GPU Memory Check

In [ ]:
if torch.cuda.is_available():
    print("🎮 GPU Memory Status:")
    print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"   Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
else:
    print("💻 Running on CPU")

## 9. Next Steps

✅ **If all tests pass, you're ready to:**

1. **Generate density maps** (if not done):
   ```bash
   python ml/src/csrnet/training/generate_density_maps.py
   ```

2. **Start training**:
   ```bash
   python ml/src/csrnet/training/train.py
   ```

3. **Monitor training**:
   ```bash
   tensorboard --logdir ml/src/csrnet/training/logs/tensorboard
   ```

4. **Evaluate model**:
   ```bash
   python ml/src/csrnet/training/evaluate.py --checkpoint ml/fine-tunned/csrnet/csrnet_best.pth --visualize
   ```

## Troubleshooting

### Common Issues:

1. **"Density maps not found"**
   - Run: `python ml/src/csrnet/training/generate_density_maps.py`

2. **"CUDA out of memory"**
   - Reduce batch_size in `ml/csrnet_config.yaml`

3. **"Dataset not found"**
   - Check paths in config file
   - Verify ShanghaiTech dataset is in correct location

4. **Import errors**
   - Make sure you're running from project root
   - Check if all dependencies are installed